# **US Medical Insurance Costs Analysis**
## **1. Introduction**
Medical insurance costs vary considerably between individuals depending on demographic characteristics, lifestyle factors, and geographic location. Understanding these variations can help identify patterns in insurance charges and the factors associated with differences in healthcare costs.

This project uses a publicly available U.S. medical insurance dataset with one row per individual. It records age, sex, body mass index (BMI), number of children, smoking status, region, and individual medical insurance charges.

The analysis follows a structured workflow: data inspection, cleaning, standardization, and quality validation (this notebook), followed by exploratory data analysis and regression modeling in later notebooks.

## **2. Objective**
The objective of this project is to analyze individual medical insurance charges using Python and to build a reproducible analysis workflow.

Specifically, the project will:
- Inspect the dataset's structure, variables, data types, and overall quality.
- Identify and address data quality issues, including missing values, duplicate records, inconsistent categorical values, and invalid values.
- Standardize column names and categorical values.
- Explore relationships between insurance charges and individual characteristics.
- Build and evaluate regression models that estimate charges from the available features.
- Communicate findings through clear visualizations, statistical summaries, and documented conclusions.

## **3. Problem Statement**
Insurance charges differ widely between individuals, but how much of that difference is associated with demographic characteristics, lifestyle factors, and region needs systematic investigation.

This project asks how charges vary across the available features, and whether those features can help explain and predict that variation. The analysis addresses these questions:
1. How are insurance charges distributed across the dataset?
2. How are age and BMI associated with charges?
3. How do charges differ between smokers and non-smokers?
4. Do charges vary by region or sex?
5. How accurately can regression models estimate charges from the available features?

**Scope and limitations:** The findings describe associations in this dataset, not causal effects. Model results are an analytical exercise and not a production-ready pricing solution.

## **4. Data Loading**
The raw data is loaded from `data/raw/insurance.csv`. The original DataFrame (`df_raw`) is kept unchanged for comparison, and all cleaning is applied to a working copy (`df`).

| Variable | Type | Description |
|---|---|---|
| `age` | Numeric | Age of the primary beneficiary (years) |
| `sex` | Categorical | Sex of the beneficiary (female, male) |
| `bmi` | Numeric | Body mass index (kg/m²) |
| `children` | Numeric | Number of dependents covered by the insurance |
| `smoker` | Categorical | Smoking status (yes, no) |
| `region` | Categorical | Residential region in the US (northeast, northwest, southeast, southwest) |
| `charges` | Numeric | Individual medical costs billed by health insurance (target variable) |

In [18]:
# Import pandas for working with tabular data.
import pandas as pd

# Import Path to construct a file path in a platform-independent way.
from pathlib import Path

# Define the path to the raw US medical insurance data file.
data_path = Path("../data/raw/insurance.csv")

# Load the original dataset
df_raw = pd.read_csv(data_path)

# Create a working copy for cleaning and preparation
df = df_raw.copy()

## **5. Data Inspection**
The structure, data types, and summary statistics were reviewed before any cleaning.

In [19]:
# Display the first few rows of the DataFrame to preview the data.
print("First 10 rows of the dataset:")
df.head(10)

First 10 rows of the dataset:


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
5,31,female,25.740,0,no,southeast,3756.62160
6,46,female,33.440,1,no,southeast,8240.58960
7,37,female,27.740,3,no,northwest,7281.50560
8,37,male,29.830,2,no,northeast,6406.41070
9,60,female,25.840,0,no,northwest,28923.13692


In [20]:
print("Statistical summary of the dataset:")
df.describe(include="all").T

Statistical summary of the dataset:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,1338.0,NaN,NaN,NaN,39.207025,14.04996,18.0,27.0,39.0,51.0,64.0
sex,1338,2,male,676,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bmi,1338.0,NaN,NaN,NaN,30.663397,6.098187,15.96,26.29625,30.4,34.69375,53.13
children,1338.0,NaN,NaN,NaN,1.094918,1.205493,0.0,0.0,1.0,2.0,5.0
smoker,1338,2,no,1064,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,1338,4,southeast,364,NaN,NaN,NaN,NaN,NaN,NaN,NaN
charges,1338.0,NaN,NaN,NaN,13270.422265,12110.011237,1121.8739,4740.28715,9382.033,16639.912515,63770.42801


In [21]:
print("Dataset shape:", df.shape)

Dataset shape: (1338, 7)


In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


**Observations:**
- The dataset has 1,338 rows and 7 columns , with no missing values.
- Data types match what each variable represents: `age` and `children` are integers, `bmi` and `charges` are floats, and `sex`, `smoker`, and `region` are text.
- `sex` has two categories (female, male), `smoker` has two (yes, no), and `region` has four (northeast, northwest, southeast, southwest).
- The summary statistics show no impossible values at first glance (for example, no negative ages or charges). The mean of `charges` is well above its median, which suggests a right-skewed distribution.
- Outliers are not assessed at this stage. `bmi` and `charges` will be examined with boxplots and the IQR rule in the exploratory analysis.

## **6. Data Cleaning and Preparation**

### 6.1 Standardizing text values
Leading and trailing whitespace was removed from the categorical columns (`sex`, `smoker`, `region`). The cleaned values were compared with the raw values to confirm how many entries changed, if any.

In [23]:
categorical_columns = ["sex", "smoker", "region"]

for col in categorical_columns:
    df[col] = df[col].str.strip()

In [24]:
for col in categorical_columns:
    print(f"{col}: {df[col].unique()}")

sex: <StringArray>
['female', 'male']
Length: 2, dtype: str
smoker: <StringArray>
['yes', 'no']
Length: 2, dtype: str
region: <StringArray>
['southwest', 'southeast', 'northwest', 'northeast']
Length: 4, dtype: str


In [25]:
categorical_columns = ["sex", "smoker", "region"]

for col in categorical_columns:
    comparison = pd.DataFrame({
        "raw": df_raw[col],
        "cleaned": df[col]
    })

    changed = comparison["raw"] != comparison["cleaned"]

    print(f"\n{col}: {changed.sum()} values changed")
    display(comparison.loc[changed])


sex: 0 values changed


,raw,cleaned



smoker: 0 values changed


,raw,cleaned



region: 0 values changed


,raw,cleaned


### 6.2 Duplicate records
Duplicate rows were identified and inspected before removal. Exact duplicates add no new information and can bias summary statistics and model evaluation, so they were dropped and the index was reset. The row count after this step is reported below.

In [26]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

if duplicate_count > 0:
    display(df.loc[df.duplicated(keep=False)].sort_values(by=df.columns.tolist()))

Duplicate rows: 1


,age,sex,bmi,children,smoker,region,charges
195,19,male,30.59,0,no,northwest,1639.5631
581,19,male,30.59,0,no,northwest,1639.5631


In [27]:
df = df.drop_duplicates().reset_index(drop=True)

In [28]:
print("Dataset shape:", df.shape)

Dataset shape: (1337, 7)


### 6.3 Column naming
Columns were renamed for clarity and consistency:
- `children` → `no_of_children`
- `charges` → `insurance_charges`

In [29]:
df.columns = df.columns.str.strip().str.lower()
df = df.rename(columns={
    "children": "no_of_children",
    "charges": "insurance_charges"
})

In [30]:
df.head()

,age,sex,bmi,no_of_children,smoker,region,insurance_charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


### 6.4 Saving the cleaned dataset
The prepared dataset is saved to `data/cleaned/insurance_cleaned.csv` so that later notebooks can load it directly instead of repeating the cleaning steps. The raw file in `data/raw/` is not modified.

In [35]:
# Define the output path
cleaned_data_path = Path("../data/cleaned/insurance_cleaned.csv")

# Create the directory if it doesn't exist
cleaned_data_path.parent.mkdir(parents=True, exist_ok=True)

# Save the prepared DataFrame to CSV
df.to_csv(cleaned_data_path, index=False)

print(f"Cleaned data saved to: {cleaned_data_path}")

Cleaned data saved to: ..\data\cleaned\insurance_cleaned.csv


### 6.5 Summary of the cleaned dataset
- Rows after cleaning: (1,337 if one duplicate was removed)
- Missing values: 
- Categorical values standardized: `sex`, `smoker`, `region`
- The cleaned dataset is saved to `data/cleaned/` for use in the next notebook. 